# Lab10 — Tool-Governed AI Agent Simulation

Goal:
Understand how an AI-agent system can separate planning, tool access, permission checks, governance decisions, and execution.

Main focus:
- AI agent orchestration
- tool registry
- permission layer
- execution governance
- trust boundaries
- observability log

In [1]:
import sys
print(sys.executable)

/mnt/c/Users/ggaru/ai-projects/rag-security-review-lab/.venv/bin/python3


## Step 1 — Tool Registry

The orchestration layer maintains a controlled registry of available tools.

The model may request tools,
but execution authority belongs to the governance layer.

In [1]:
tools = {
    "web_search": {
        "risk": "low",
        "requires_approval": False
    },

    "send_email": {
        "risk": "high",
        "requires_approval": True
    },

    "read_file": {
        "risk": "medium",
        "requires_approval": True
    }
}

In [2]:
for tool_name, tool_info in tools.items():
    print(f"\nTOOL: {tool_name}")
    print(f"Risk level: {tool_info['risk']}")
    print(f"Requires approval: {tool_info['requires_approval']}")


TOOL: web_search
Risk level: low
Requires approval: False

TOOL: send_email
Risk level: high
Requires approval: True

TOOL: read_file
Risk level: medium
Requires approval: True


## Step 2 — Permission Decision Layer

The governance layer evaluates requested actions before execution.

The model may suggest an action,
but the orchestration system decides whether execution is allowed,
blocked, or requires approval.

In [4]:
def evaluate_tool_request(tool_name):

    tool = tools.get(tool_name)

    if tool is None:
        return "DENY"

    if tool["requires_approval"]:
        return "REQUIRE_APPROVAL"

    return "ALLOW"

In [5]:
requests = [
    "web_search",
    "send_email",
    "read_file",
    "delete_database"
]

for request in requests:

    decision = evaluate_tool_request(request)

    print(f"\nRequested tool: {request}")
    print(f"Governance decision: {decision}")


Requested tool: web_search
Governance decision: ALLOW

Requested tool: send_email
Governance decision: REQUIRE_APPROVAL

Requested tool: read_file
Governance decision: REQUIRE_APPROVAL

Requested tool: delete_database
Governance decision: DENY


## Step 3 — Human-in-the-Loop Approval

High-risk actions should require explicit human approval before execution.

The orchestration layer separates model reasoning, governance evaluation, and final execution authority.

In [7]:

def process_request(tool_name):

    decision = evaluate_tool_request(tool_name)

    print(f"\nRequested tool: {tool_name}")
    print(f"Governance decision: {decision}")

    if decision == "ALLOW":
        print("Execution approved.")

    elif decision == "REQUIRE_APPROVAL":
        print("Human approval required.")

    else:
        print("Execution blocked.")

In [8]:
test_requests = [
    "web_search",
    "send_email",
    "delete_database"
]

for request in test_requests:
    process_request(request)


Requested tool: web_search
Governance decision: ALLOW
Execution approved.

Requested tool: send_email
Governance decision: REQUIRE_APPROVAL
Human approval required.

Requested tool: delete_database
Governance decision: DENY
Execution blocked.


## Step 4 — Execution Boundary

The model may suggest actions, but execution only occurs after governance approval.

Execution authority belongs to the orchestration layer, not the language model itself.

In [11]:
def execute_tool(tool_name):

    decision = evaluate_tool_request(tool_name)

    print(f"\nRequested tool: {tool_name}")
    print(f"Governance decision: {decision}")

    if decision == "ALLOW":

        print("Tool execution started.")
        print(f"{tool_name} executed successfully.")

    elif decision == "REQUIRE_APPROVAL":

        print("Execution paused.")
        print("Waiting for human approval.")

    else:

        print("Execution blocked by governance layer.")

In [12]:
execution_requests = [
    "web_search",
    "send_email",
    "delete_database"
]

for request in execution_requests:
    execute_tool(request)


Requested tool: web_search
Governance decision: ALLOW
Tool execution started.
web_search executed successfully.

Requested tool: send_email
Governance decision: REQUIRE_APPROVAL
Execution paused.
Waiting for human approval.

Requested tool: delete_database
Governance decision: DENY
Execution blocked by governance layer.


## Step 5 — Observability Logging

The orchestration layer should preserve visibility into governance decisions and execution behavior.

Observability enables:
auditability,
decision reconstruction,
and operational monitoring.

In [14]:
execution_log = []

In [17]:

def log_event(tool_name, decision):

    event = {
        "tool": tool_name,
        "decision": decision
    }
    log_event(tool_name, decision)
    execution_log.append(event)
    

In [18]:
from datetime import datetime

execution_log = []

def log_event(tool_name, decision, status):
    event = {
        "timestamp": str(datetime.now()),
        "tool": tool_name,
        "decision": decision,
        "status": status
    }

    execution_log.append(event)

In [20]:
from datetime import datetime

execution_log = []

def log_event(tool_name, reason, action, execution_status):
    event = {
        "timestamp": str(datetime.now()),
        "tool": tool_name,
        "reason": reason,
        "action": action,
        "execution_status": execution_status
    }

    execution_log.append(event)

In [21]:
log_event("email_tool", "policy_violation", "blocked", "skipped")

print(execution_log)

[{'timestamp': '2026-05-18 09:34:49.929995', 'tool': 'email_tool', 'reason': 'policy_violation', 'action': 'blocked', 'execution_status': 'skipped'}]


In [22]:
log_event("search_tool", "low_risk", "approved", "executed")
log_event("payment_tool", "high_risk_transaction", "escalated", "human_review")

print(execution_log)

[{'timestamp': '2026-05-18 09:34:49.929995', 'tool': 'email_tool', 'reason': 'policy_violation', 'action': 'blocked', 'execution_status': 'skipped'}, {'timestamp': '2026-05-18 09:39:05.379475', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 09:39:05.379513', 'tool': 'payment_tool', 'reason': 'high_risk_transaction', 'action': 'escalated', 'execution_status': 'human_review'}]


In [1]:
from datetime import datetime

execution_log = []

def log_event(tool_name, reason, action, execution_status):
    event = {
        "timestamp": str(datetime.now()),
        "tool": tool_name,
        "reason": reason,
        "action": action,
        "execution_status": execution_status
    }

    execution_log.append(event)


def evaluate_tool(tool_name, risk_level):

    if risk_level == "low":
        log_event(
            tool_name,
            "low_risk",
            "approved",
            "executed"
        )
        return "approved"
    elif risk_level == "medium":
        log_event(
            tool_name,
            "medium_risk",
            "escalated",
            "human_review"
        )
        return "escalated"
    elif risk_level == "high":
        log_event(
            tool_name,
            "high_risk",
            "denied",
            "skipped"
        )
        return "denied"

In [3]:
decision = evaluate_tool("search_tool", "low")
print(decision)

approved


In [26]:
evaluate_tool("search_tool", "low")
evaluate_tool("payment_tool", "medium")
evaluate_tool("email_tool", "high")

print(execution_log)

[{'timestamp': '2026-05-18 09:54:33.789572', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 09:54:33.789598', 'tool': 'payment_tool', 'reason': 'medium_risk', 'action': 'escalated', 'execution_status': 'human_review'}, {'timestamp': '2026-05-18 09:54:33.789633', 'tool': 'email_tool', 'reason': 'high_risk', 'action': 'denied', 'execution_status': 'skipped'}]


In [5]:
def run_tool(tool_name):
    print(f"Running {tool_name}")

In [6]:
decision = evaluate_tool("search_tool", "low")

if decision == "approved":
    run_tool("search_tool")

Running search_tool


In [32]:
def run_tool(tool_name):
    print(f"Running {tool_name}")


decision = evaluate_tool("search_tool", "low")

if decision == "approved":
    run_tool("search_tool")

In [11]:
decision = evaluate_tool("email_tool", "high")

if decision == "approved":
    run_tool("email_tool")

In [7]:
print(execution_log)

[{'timestamp': '2026-05-18 11:03:57.149558', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 11:06:04.806910', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 11:14:43.377466', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 11:16:43.757222', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}]


In [9]:
decision = evaluate_tool("email_tool", "high")

if decision == "approved":
    run_tool("email_tool")
else:
    print("Tool was not executed")

Tool was not executed


In [12]:
print(execution_log)

[{'timestamp': '2026-05-18 11:03:57.149558', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 11:06:04.806910', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 11:14:43.377466', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 11:16:43.757222', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 11:38:49.303089', 'tool': 'email_tool', 'reason': 'high_risk', 'action': 'denied', 'execution_status': 'skipped'}, {'timestamp': '2026-05-18 11:39:49.033952', 'tool': 'email_tool', 'reason': 'high_risk', 'action': 'denied', 'execution_status': 'skipped'}]


In [2]:
decision = evaluate_tool("search_tool", "low")
print(decision)

approved


In [13]:
decision = evaluate_tool("search_tool", "low")
decision = evaluate_tool("payment_tool", "medium")
decision = evaluate_tool("email_tool", "high")

print(execution_log)

[{'timestamp': '2026-05-18 11:03:57.149558', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 11:06:04.806910', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 11:14:43.377466', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 11:16:43.757222', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}, {'timestamp': '2026-05-18 11:38:49.303089', 'tool': 'email_tool', 'reason': 'high_risk', 'action': 'denied', 'execution_status': 'skipped'}, {'timestamp': '2026-05-18 11:39:49.033952', 'tool': 'email_tool', 'reason': 'high_risk', 'action': 'denied', 'execution_status': 'skipped'}, {'timestamp': '2026-05-18 11:43:29.078855', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'

In [14]:
for event in execution_log:
    print(event)

{'timestamp': '2026-05-18 11:03:57.149558', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}
{'timestamp': '2026-05-18 11:06:04.806910', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}
{'timestamp': '2026-05-18 11:14:43.377466', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}
{'timestamp': '2026-05-18 11:16:43.757222', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}
{'timestamp': '2026-05-18 11:38:49.303089', 'tool': 'email_tool', 'reason': 'high_risk', 'action': 'denied', 'execution_status': 'skipped'}
{'timestamp': '2026-05-18 11:39:49.033952', 'tool': 'email_tool', 'reason': 'high_risk', 'action': 'denied', 'execution_status': 'skipped'}
{'timestamp': '2026-05-18 11:43:29.078855', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}
{'tim

In [15]:
for event in execution_log[-3:]:
    print(event)

{'timestamp': '2026-05-18 11:43:29.078855', 'tool': 'search_tool', 'reason': 'low_risk', 'action': 'approved', 'execution_status': 'executed'}
{'timestamp': '2026-05-18 11:43:29.078897', 'tool': 'payment_tool', 'reason': 'medium_risk', 'action': 'escalated', 'execution_status': 'human_review'}
{'timestamp': '2026-05-18 11:43:29.078916', 'tool': 'email_tool', 'reason': 'high_risk', 'action': 'denied', 'execution_status': 'skipped'}


## Lab Summary

This lab simulated a governed AI-agent workflow with:

- tool registry;
- permission evaluation;
- execution control;
- Human-in-the-loop approval;
- execution boundaries;
- observability logging.

Key governance distinction:

Model reasoning does not automatically imply execution authority.

The orchestration layer controls:
authorization,
execution,
and operational visibility.